In [ ]:
import os
import sys

# Assuming the notebook is inside /notebooks, move one level up
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)

print("Current working directory:", os.getcwd())

In [ ]:
# !pip install ultralytics opencv-python torch torchvision torchaudio roboflow supervision scikit-learn

In [ ]:
from roboflow import download_dataset
from ultralytics import YOLO


WORKSPACE = "tutorial-extpj"              # Found in the URL
PROJECT = "brain-tumor-detection-1zpnr"   # Found in the URL
VERSION = 5                            # Matches the dataset version you downloaded

dataset_url = f"https://universe.roboflow.com/{WORKSPACE}/{PROJECT}/dataset/{VERSION}"

### Available YOLOv8 Models

| Model                | Size    | Speed               | Accuracy         | Best Use Case                                      |
|----------------------|---------|--------------------|----------------|---------------------------------------------------|
| `yolov8n.pt`</br ><span style="white-space: nowrap;">`yolov8n-seg.pt`</span> | Nano    | ⚡⚡⚡⚡⚡ (Fastest) | ⭐ (Lowest)     | Fast inference & segmentation, low-resource devices |
| `yolov8s.pt`</br > `yolov8s-seg.pt` | Small   | ⚡⚡⚡⚡        | ⭐⭐              | Good balance of speed & accuracy, recommended for most tasks |
| `yolov8m.pt`</br > `yolov8m-seg.pt` | Medium  | ⚡⚡⚡         | ⭐⭐⭐             | Higher accuracy, more computational power needed  |
| `yolov8l.pt`</br > `yolov8l-seg.pt` | Large   | ⚡⚡          | ⭐⭐⭐⭐            | High accuracy, slower inference, best for complex datasets |
| `yolov8x.pt`</br > `yolov8x-seg.pt` | X-Large | ⚡           | ⭐⭐⭐⭐⭐ (Highest) | Best accuracy, very slow inference, needs strong hardware |


In [ ]:
dataset = download_dataset(dataset_url, 'yolov8', location="data")
def train_object_detection_model():
    model = YOLO("../yolov8s.pt")

    results = model.train(
        data=f'{dataset.location}/data.yaml', 
        epochs=150,
        patience=20,
        verbose=False
    )
    metrics = model.val()
    print(metrics)

In [ ]:
train_object_detection_model()

53 epochs completed in 21.243 hours. 1275 mins.
New laptop: 83 epochs completed in 19h 37 mins.

## Resume Training the Next Day

In [ ]:
# model = YOLO("runs/detect/train2/weights/last.pt")  # Load the last checkpoint

# results = model.train(resume=True) 

In [ ]:
import os
from IPython.display import Image

HOME = os.getcwd()
print(HOME)

Image(filename=f'{HOME}/runs/detect/train42/confusion_matrix.png', width=1000)

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/results.png', width=1000)

In [ ]:
import supervision as sv

ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/valid/images",
    annotations_directory_path=f"{dataset.location}/valid/labels",
    data_yaml_path=f"{dataset.location}/data.yaml"
)

ds.classes

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

def get_best_model(runs_dir='runs/detect', metric='metrics/mAP50(B)'):
    """
    This function scans all the training folders in the specified directory,
    prints the value of the specified metric for each run, and returns the best YOLOv8 model
    directly.

    Parameters:
    - runs_dir: str, the directory containing the training folders (default is 'runs/detect').
    - metric: str, the metric to compare (default is 'metrics/mAP50(B)').

    Returns:
    - best_model: YOLO model object, the best YOLOv8 model based on the selected metric.

    Available Metrics:
    - 'metrics/precision(B)'
    - 'metrics/recall(B)'
    - 'metrics/mAP50(B)'
    - 'metrics/mAP50-95(B)'
    - 'val/box_loss'
    - 'val/cls_loss'
    - 'val/dfl_loss'
    - 'train/box_loss'
    - 'train/cls_loss'
    - 'train/dfl_loss'
    - 'lr/pg0'
    - 'lr/pg1'
    - 'lr/pg2'
    """
    
    best_metric_value = -1
    best_model = None
    best_folder_path = None
    all_metrics = set()

    # Loop through each subfolder in the 'runs/detect' directory
    for folder in os.listdir(runs_dir):
        folder_path = os.path.join(runs_dir, folder)
        
        # Check if this is a folder and it contains a results.csv
        if os.path.isdir(folder_path) and 'results.csv' in os.listdir(folder_path):
            results_file = os.path.join(folder_path, 'results.csv')
            
            # Load the CSV file into a DataFrame
            df = pd.read_csv(results_file)
            
            # Add all column names (metrics) to the all_metrics set
            all_metrics.update(df.columns)
            
            # Check if the specified metric is in the columns
            if metric in df.columns:
                # Get the last epoch's value for the selected metric
                last_metric_value = df[metric].iloc[-1]
                
                # Print the folder name and the corresponding metric value
                print(f"Folder: {folder} - {metric}: {last_metric_value}")
                
                # Check if this is the best metric value so far
                if last_metric_value > best_metric_value:
                    best_metric_value = last_metric_value
                    best_folder_path = folder_path.replace("\\", "/")
                    best_model_path = os.path.join(folder_path, 'weights', 'best.pt').replace("\\", "/")
                    
                    # Load the best model
                    best_model = YOLO(best_model_path)
    
    return best_model, best_folder_path

# model, best_folder_path = get_best_model(runs_dir='runs/detect', metric='metrics/mAP50(B)')
# best_folder_path


In [ ]:
import torch
from torch.nn.modules.container import Sequential, ModuleList
from ultralytics.nn.tasks import DetectionModel
from torch.nn.modules.conv import Conv2d
from torch.nn.modules.batchnorm import BatchNorm2d
from torch.nn.modules.activation import SiLU
from ultralytics.nn.modules.block import C2f, Bottleneck, SPPF
from torch.nn.modules.pooling import MaxPool2d
from torch.nn.modules.upsampling import Upsample
from ultralytics.nn.modules.conv import Concat
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.modules.block import DFL  # Import DFL

# Add necessary safe globals before loading the model
torch.serialization.add_safe_globals([DetectionModel, Sequential, ModuleList, Conv2d, BatchNorm2d, SiLU, C2f, Bottleneck, SPPF, MaxPool2d, Upsample, Concat, Detect, DFL])

# Now attempt to load the model
model = YOLO("../runs/detect/train4/weights/best.pt")


In [ ]:
from supervision.metrics import MeanAveragePrecision
%matplotlib inline

predictions = []
targets = []

for _, image, target in ds:
    results = model(image, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    predictions.append(detections)
    targets.append(target)

map = MeanAveragePrecision().update(predictions, targets).compute()
print("mAP 50:95", map.map50_95)
print("mAP 50", map.map50)
print("mAP 75", map.map75)

In [ ]:
map.plot()

In [ ]:
import supervision as sv

ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/valid/images",
    annotations_directory_path=f"{dataset.location}/valid/labels",
    data_yaml_path=f"{dataset.location}/data.yaml"
)

In [ ]:
import random

i = random.randint(0, len(ds) - 1)
image_path, image, target = ds[i]

results = model(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results).with_nms()

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = image.copy()
annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

true_labels = ', '.join([model.names[int(label)] for label in target.class_id])
predicted_labels = ', '.join([model.names[int(label)] for label in detections.class_id])

print(f"True Labels: {true_labels}")
print(f"Predicted Labels: {predicted_labels}")
sv.plot_image(annotated_image)

In [ ]:
model.names

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Prepare lists for true and predicted labels
y_true = []
y_pred = []
i = 0
# Iterate over all images in the test dataset
for image_path, image, target in ds:

    results = model(image, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results).with_nms()

    i += 1
    if not len(target.class_id):
        print(f"{i}, {target.class_id}")
    true_label = [model.names[int(label)] for label in target.class_id][0]
    predicted_labels = [model.names[int(label)] for label in detections.class_id]
    pred_label = true_label if true_label in predicted_labels \
        else "notumor" if not len (predicted_labels) else predicted_labels[0]

    y_true.append(true_label)
    y_pred.append(pred_label)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
labels = list(model.names.values())

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm

In [ ]:
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=model.names.values(), 
    yticklabels=model.names.values()
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels');

In [ ]:
print(classification_report(y_true, y_pred, labels=labels))

## Upload new model to Roboflow

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from roboflow import Roboflow

def get_api_key():
    load_dotenv()
    api_key = os.getenv("ROBOFLOW_API_KEY")
    if not api_key:
        raise ValueError("API Key not found! Make sure the .env file is set up correctly.")
    return api_key

def get_versions_df(project):
    version_info = []
    for version in project.versions():
        version_info.append({
            "Version": version.version,
            "Images": version.images,
            "Training Images": version.splits['train'],
            "Validation Images": version.splits['valid'],
            "Test Images": version.splits['test'],
            "Public": version.public,
            "Model": 'Yes' if version.model else 'No'
        })

    return pd.DataFrame(version_info)

In [ ]:
# Create a list to store version information

rf = Roboflow(api_key=get_api_key())
project = rf.workspace(WORKSPACE).project(PROJECT)

versions_df = get_versions_df(project)
versions_df

In [ ]:
# best_folder_path

In [ ]:
# !pip install ultralytics==8.0.196
# !pip install ultralytics --upgrade

In [ ]:
# Upload the model to Roboflow
model_version = project.version(VERSION).deploy(
    model_type="yolov8", 
    model_path="../runs/detect/train"
)

print(f"✅ Model uploaded successfully! New version: {model_version}")

In [ ]:
# Check if new model uploaded
project = rf.workspace(WORKSPACE).project(PROJECT)

versions_df = get_versions_df(project)
versions_df